# Module 4.3 — Retrieval That Actually Works
**DeskMate SLM Curriculum · Phase 4 · Notebook 23**

Build and evaluate four retrieval variants:

| Variant | Method |
|---|---|
| A | Dense-only (FAISS / bge-small) |
| B | BM25-only (rank_bm25) |
| C | Hybrid: RRF(dense, BM25) |
| D | Hybrid + cross-encoder reranker |

Measured by hit-rate@1/3/5 and MRR on a 20-query gold set.

Read `4.3_retrieval.md` for full theory.

---

## Step 0 — Install

In [ ]:
%%capture
!pip install -q sentence-transformers==3.1.1 faiss-cpu==1.8.0 chromadb==0.5.5 \
               rank-bm25==0.2.2 pickle5

In [ ]:
import re, json, pathlib, pickle, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
print('Imports OK')

## Step 1 — Runtime & Paths

In [ ]:
try:
    import google.colab; RUNTIME = 'colab'
    PROJECT_ROOT = pathlib.Path('/content/slm')
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = 'kaggle'
        PROJECT_ROOT = pathlib.Path('/kaggle/working/slm')
    except ImportError:
        RUNTIME = 'local'
        PROJECT_ROOT = pathlib.Path('.').resolve()

DATA_PROC  = PROJECT_ROOT / 'data' / 'processed'
FAISS_DIR  = PROJECT_ROOT / 'data' / 'faiss'
CHROMA_DIR = PROJECT_ROOT / 'data' / 'chroma_db'
REPORTS    = PROJECT_ROOT / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
print(f'Runtime: {RUNTIME}')

## Step 2 — Load Assets from Module 4.2

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

# Chunk records
records_path = DATA_PROC / 'chunk_records.jsonl'
if records_path.exists():
    with open(records_path) as f:
        chunk_records = [json.loads(l) for l in f]
    print(f'Loaded {len(chunk_records)} chunk records')
else:
    raise FileNotFoundError(
        'chunk_records.jsonl not found — run Module 4.2 (notebook 22) first')

# FAISS index
faiss_path = FAISS_DIR / 'deskmate_faq.index'
if faiss_path.exists():
    faiss_index = faiss.read_index(str(faiss_path))
    print(f'FAISS index: {faiss_index.ntotal} vectors')
else:
    raise FileNotFoundError('FAISS index not found — run Module 4.2 first')

# Embedding model
BGE_PREFIX = 'Represent this sentence for searching relevant passages: '
embedder   = SentenceTransformer('BAAI/bge-small-en-v1.5')
print('Embedding model loaded')

## Step 3 — Build BM25 Index

In [ ]:
from rank_bm25 import BM25Okapi

corpus_tokens = [r['text'].lower().split() for r in chunk_records]
bm25          = BM25Okapi(corpus_tokens)
print(f'BM25 index built: {len(corpus_tokens)} documents')

# Quick test
test_query = 'password reset email not received'
scores = bm25.get_scores(test_query.lower().split())
top3   = scores.argsort()[::-1][:3]
print(f'\nBM25 top-3 for: "{test_query}"')
for rank, idx in enumerate(top3):
    r = chunk_records[idx]
    print(f'  [{rank+1}] score={scores[idx]:.3f}  src={r["source"]}')
    print(f'       {r["text"][:100]}...')

## Step 4 — Dense Retrieval (FAISS)

In [ ]:
def dense_retrieve(query, k=20, product_filter=None):
    q_emb = embedder.encode(BGE_PREFIX + query, normalize_embeddings=True)
    D, I  = faiss_index.search(np.array([q_emb], dtype='float32'), k=k)
    results = []
    for score, idx in zip(D[0], I[0]):
        if idx < 0: continue
        r = chunk_records[idx]
        if product_filter and r['product'] not in (product_filter, 'all'):
            continue
        results.append({'chunk': r, 'score': float(score), 'idx': int(idx)})
    return results

test_results = dense_retrieve('password reset email not received', k=3)
print('Dense top-3 for: "password reset email not received"')
for i, r in enumerate(test_results):
    print(f'  [{i+1}] score={r["score"]:.3f}  src={r["chunk"]["source"]}')
    print(f'       {r["chunk"]["text"][:100]}...')

## Step 5 — Where BM25 Wins vs Dense Wins

Test on queries that favour each approach.

In [ ]:
CONTRAST_QUERIES = [
    ('ERR_404 export error',       'exact keyword — BM25 should win'),
    ('cannot access my workspace', 'semantic — dense should win'),
    ('TOTP 2FA backup codes',      'acronym — BM25 should win'),
    ('service not working for hours', 'semantic — dense should win'),
]

print(f'{"Query":<45} {"Expected":>22} {"Dense top-1":>20} {"BM25 top-1":>20}')
print('-' * 110)
for q, expected in CONTRAST_QUERIES:
    d_top  = dense_retrieve(q, k=1)
    d_src  = d_top[0]['chunk']['section'] if d_top else 'N/A'
    b_toks = q.lower().split()
    b_scores = bm25.get_scores(b_toks)
    b_top_idx = int(b_scores.argsort()[::-1][0])
    b_src = chunk_records[b_top_idx]['section']
    print(f'{q:<45} {expected:>22} {d_src:>20} {b_src:>20}')

## Step 6 — Reciprocal Rank Fusion

In [ ]:
def rrf(ranked_id_lists, k=60):
    scores = {}
    for ranked in ranked_id_lists:
        for rank, doc_id in enumerate(ranked):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

def bm25_retrieve_ids(query, k=20):
    toks   = query.lower().split()
    scores = bm25.get_scores(toks)
    return [chunk_records[i]['id'] for i in scores.argsort()[::-1][:k]]

def hybrid_retrieve(query, k_final=3, product_filter=None):
    dense_hits = dense_retrieve(query, k=20, product_filter=product_filter)
    dense_ids  = [h['chunk']['id'] for h in dense_hits]
    bm25_ids   = bm25_retrieve_ids(query, k=20)
    merged_ids = rrf([dense_ids, bm25_ids])
    id_to_chunk = {r['id']: r for r in chunk_records}
    results = [id_to_chunk[cid] for cid in merged_ids if cid in id_to_chunk]
    return results[:k_final]

hybrid_results = hybrid_retrieve('ERR_404 export error', k_final=3)
print('Hybrid top-3 for: "ERR_404 export error"')
for i, r in enumerate(hybrid_results):
    print(f'  [{i+1}] src={r["source"]}  section={r["section"]}')
    print(f'       {r["text"][:100]}...')

## Step 7 — Cross-Encoder Reranker

In [ ]:
from sentence_transformers import CrossEncoder

print('Loading cross-encoder reranker ...')
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('Reranker loaded.')

def rerank(query, candidates, k=3):
    if not candidates: return candidates
    pairs  = [(query, c['text']) for c in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    return [c for _, c in ranked[:k]]

def full_retrieve(query, fields=None, k_final=3):
    product_filter = (fields or {}).get('product')
    # Hybrid (top-20 candidates)
    dense_hits = dense_retrieve(query, k=20, product_filter=product_filter)
    dense_ids  = [h['chunk']['id'] for h in dense_hits]
    bm25_ids   = bm25_retrieve_ids(query, k=20)
    merged_ids = rrf([dense_ids, bm25_ids])
    id_to_chunk = {r['id']: r for r in chunk_records}
    candidates  = [id_to_chunk[cid] for cid in merged_ids
                   if cid in id_to_chunk][:20]
    # Rerank
    return rerank(query, candidates, k=k_final)

result = full_retrieve('ERR_404 export error',
                        fields={'product': 'DeskMate Pro'})
print('Hybrid + rerank top-3 for: "ERR_404 export error"')
for i, r in enumerate(result):
    print(f'  [{i+1}] src={r["source"]}  section={r["section"]}')
    print(f'       {r["text"][:100]}...')

## Step 8 — Gold Query Set (20 Queries)

In [ ]:
# (query, list of gold source filenames that should appear in top-k)
GOLD_QUERIES = [
    # account_access
    ('How do I reset my password?',          ['account_login.md']),
    ('My account is locked after failed logins', ['account_login.md']),
    ('How does two-factor authentication work?', ['account_login.md']),
    ('Can I use SSO with DeskMate?',          ['account_login.md']),
    ('password reset link not arriving',      ['account_login.md']),
    # billing_dispute
    ('What is the refund policy?',            ['billing_refunds.md']),
    ('I was charged twice for my subscription', ['billing_refunds.md']),
    ('How do I download my invoice?',         ['billing_refunds.md']),
    ('Can I change my plan mid-month?',       ['billing_refunds.md']),
    ('30 day money back guarantee',           ['billing_refunds.md']),
    # technical_bug
    ('CSV export returns 500 error',          ['technical_bugs.md']),
    ('ERR_404 on the export button',          ['technical_bugs.md']),
    ('mobile app crash on iOS',               ['technical_bugs.md']),
    ('how to report a bug',                   ['technical_bugs.md']),
    ('export dataset limit 50000 rows',       ['technical_bugs.md']),
    # usage_question
    ('How do I invite a team member?',        ['getting_started.md']),
    ('What integrations does DeskMate support?', ['getting_started.md']),
    ('Where do I find my API key?',           ['getting_started.md']),
    ('API rate limits per plan',              ['getting_started.md']),
    # outage_report
    ('Is DeskMate down right now?',           ['outage_status.md']),
]

# Normalise gold: match any chunk whose source starts with the gold filename
def gold_chunk_ids(gold_sources):
    return [
        r['id'] for r in chunk_records
        if any(r['source'].startswith(gs.replace('.md','')) for gs in gold_sources)
    ]

gold_sets = [gold_chunk_ids(g) for _, g in GOLD_QUERIES]
print(f'Gold queries: {len(GOLD_QUERIES)}')
print(f'Gold chunk IDs per query (mean): '
       f'{np.mean([len(g) for g in gold_sets]):.1f}')

## Step 9 — Evaluate All Four Retrieval Variants

In [ ]:
def hit_rate_at_k(result_id_lists, gold_id_sets, k):
    hits = sum(
        1 for res_ids, gold_ids in zip(result_id_lists, gold_id_sets)
        if any(g in res_ids[:k] for g in gold_ids)
    )
    return hits / len(result_id_lists)

def mrr_score(result_id_lists, gold_id_sets):
    rr_sum = 0.0
    for res_ids, gold_ids in zip(result_id_lists, gold_id_sets):
        for rank, doc_id in enumerate(res_ids, start=1):
            if doc_id in gold_ids:
                rr_sum += 1.0 / rank
                break
    return rr_sum / len(result_id_lists)

K_EVAL = 5   # retrieve top-K for evaluation
queries = [q for q, _ in GOLD_QUERIES]

# Variant A: dense-only
print('Evaluating Variant A: dense-only ...')
dense_ids_all = []
for q in queries:
    hits = dense_retrieve(q, k=K_EVAL)
    dense_ids_all.append([h['chunk']['id'] for h in hits])

# Variant B: BM25-only
print('Evaluating Variant B: BM25-only ...')
bm25_ids_all = [bm25_retrieve_ids(q, k=K_EVAL) for q in queries]

# Variant C: hybrid RRF
print('Evaluating Variant C: hybrid RRF ...')
hybrid_ids_all = []
for q in queries:
    res = hybrid_retrieve(q, k_final=K_EVAL)
    hybrid_ids_all.append([r['id'] for r in res])

# Variant D: hybrid + reranker
print('Evaluating Variant D: hybrid + reranker ...')
full_ids_all = []
for q in queries:
    res = full_retrieve(q, k_final=K_EVAL)
    full_ids_all.append([r['id'] for r in res])

print('Done.')

## Step 10 — Results Table

In [ ]:
variants = [
    ('A: Dense-only',         dense_ids_all),
    ('B: BM25-only',          bm25_ids_all),
    ('C: Hybrid RRF',         hybrid_ids_all),
    ('D: Hybrid + Reranker',  full_ids_all),
]

rows = []
print(f'{"Variant":<25} {"HR@1":>6} {"HR@3":>6} {"HR@5":>6} {"MRR":>6}')
print('-' * 50)
for name, id_lists in variants:
    hr1 = hit_rate_at_k(id_lists, gold_sets, k=1)
    hr3 = hit_rate_at_k(id_lists, gold_sets, k=3)
    hr5 = hit_rate_at_k(id_lists, gold_sets, k=5)
    mrr = mrr_score(id_lists, gold_sets)
    rows.append((name, hr1, hr3, hr5, mrr))
    print(f'{name:<25} {hr1:>6.2f} {hr3:>6.2f} {hr5:>6.2f} {mrr:>6.2f}')

# Plot
labels = [r[0].split(':')[0] for r in rows]
hr1s   = [r[1] for r in rows]
hr3s   = [r[2] for r in rows]
mrrs   = [r[4] for r in rows]
x = np.arange(len(labels))
width = 0.25
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - width, hr1s, width, label='HR@1', color='#4C72B0')
ax.bar(x,         hr3s, width, label='HR@3', color='#55A868')
ax.bar(x + width, mrrs, width, label='MRR',  color='#DD8452')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
ax.set_title('Retrieval Variants — HR@1, HR@3, MRR')
ax.legend(); plt.tight_layout(); plt.show()

## Step 11 — Per-Query Failure Analysis

In [ ]:
best_ids = full_ids_all   # Variant D

print('Missed queries (Variant D hybrid + reranker):')
missed = 0
for q, gold_ids, result_ids in zip(queries, gold_sets, best_ids):
    if not any(g in result_ids[:3] for g in gold_ids):
        missed += 1
        print(f'  MISS: {q}')
        print(f'    Gold  : {gold_ids[:2]}')
        print(f'    Top-3 : {result_ids[:3]}')

if missed == 0:
    print('  None — all queries hit in top-3!')
else:
    print(f'\n{missed}/{len(queries)} queries missed in top-3.')
    print('Consider: adding more chunks, adjusting chunk size, or extending the BM25 vocab.')

## Step 12 — Metadata Filter Impact

In [ ]:
# Simulate: ticket 'ERR_404 export error' with product='DeskMate Pro' extracted by encoder
q = 'ERR_404 export error'
fields_pro = {'product': 'DeskMate Pro'}
fields_none = {}

res_filtered   = full_retrieve(q, fields=fields_pro, k_final=3)
res_unfiltered = full_retrieve(q, fields=fields_none, k_final=3)

print('Without product filter:')
for r in res_unfiltered:
    print(f'  src={r["source"]}  product={r["product"]}')
print('\nWith product=DeskMate Pro filter:')
for r in res_filtered:
    print(f'  src={r["source"]}  product={r["product"]}')
print('\nFiltering leverages the Module 2.5 NER extractor — '
       'no extra labelling needed.')

## Step 13 — Save BM25 Index & Write Eval Report

In [ ]:
bm25_path = DATA_PROC / 'bm25_corpus.pkl'
with open(bm25_path, 'wb') as f:
    pickle.dump({'bm25': bm25, 'ids': [r['id'] for r in chunk_records]}, f)
print(f'BM25 index saved: {bm25_path}')

# Write retrieval eval report
lines = [
    '# Retrieval Evaluation Report\n',
    f'Gold queries: {len(GOLD_QUERIES)}  |  '
    f'Chunk corpus: {len(chunk_records)} chunks\n',
    '| Variant | HR@1 | HR@3 | HR@5 | MRR |',
    '|---|---|---|---|---|',
]
best_hr3 = 0; best_name = ''
for name, hr1, hr3, hr5, mrr in rows:
    lines.append(f'| {name} | {hr1:.2f} | {hr3:.2f} | {hr5:.2f} | {mrr:.2f} |')
    if hr3 > best_hr3:
        best_hr3 = hr3; best_name = name
lines += ['', f'## Recommendation\n',
          f'Best variant by HR@3: **{best_name}** ({best_hr3:.2f}).\n',
          'Use `full_retrieve(query, fields, k=3)` in Module 4.4.']
report_path = REPORTS / 'retrieval_eval.md'
report_path.write_text('\n'.join(lines))
print(f'Report saved: {report_path}')

## Checkpoint

> **When does pure vector search fail and BM25 save you?**

Write your answer below.

In [ ]:
answer = """
[Write your answer here]
"""
print(answer)

## Deliverable Summary

| Artifact | Location |
|---|---|
| BM25 index | `data/processed/bm25_corpus.pkl` |
| Retrieval eval report | `reports/retrieval_eval.md` |

**What you've built:** a four-variant retrieval pipeline benchmarked on a 20-query gold set.
`full_retrieve(query, fields, k=3)` is the function to wire into Module 4.4.

**Next:** Module 4.4 — Wire RAG to the decoder.
Assemble the full prompt, generate grounded replies with citations,
and measure faithfulness: is what the model said supported by what was retrieved?